In [1]:
!pip install -q transformers torch scikit-learn numpy

In [2]:
!pip install -q accelerate

In [3]:
import sys
import os
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Colab 환경 감지
IN_COLAB = 'google.colab' in sys.modules

# 1. GitHub 리포지토리 클론
if IN_COLAB and not Path("/content/kdic-crawler").exists():
    !git clone https://github.com/likelion-8/kdic-crawler.git /content/kdic-crawler

# 2. REPO_ROOT 설정
REPO_ROOT = Path("/content/kdic-crawler") if IN_COLAB else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("REPO_ROOT:", REPO_ROOT)

# 3. REPO_ROOT 하위에서 testset_all.jsonl 및 chunks_all.jsonl 위치 자동 탐색
def find_dataset_file(repo_root: Path, filename: str) -> Path:
    matches = list(repo_root.rglob(filename))
    if matches:
        return matches[0]
    # 탐색 실패 시 기본 위치 반환
    return repo_root / filename

EVAL_DATASET_PATH = find_dataset_file(REPO_ROOT, "testset_all.jsonl")
CRAWLED_DOCS_PATH = find_dataset_file(REPO_ROOT, "chunks_all.jsonl")

print("🎯 평가셋 경로:", EVAL_DATASET_PATH)
print("📚 청크셋 경로:", CRAWLED_DOCS_PATH)

# 필수 패키지 설치
!pip install -q sentence-transformers torch pandas tqdm

REPO_ROOT: /content/kdic-crawler
🎯 평가셋 경로: /content/kdic-crawler/data/testset/testset_all.jsonl
📚 청크셋 경로: /content/kdic-crawler/data/chunks_all.jsonl


In [4]:
import json
import re
from typing import List, Dict, Any

def fix_truncated_json(json_str: str) -> str:
    """잘린 따옴표 및 괄호를 추적하여 유효한 JSON으로 복구합니다."""
    quotes = re.findall(r'(?<!\\)"', json_str)
    if len(quotes) % 2 != 0:
        json_str += '"'
    
    open_braces = json_str.count('{') - json_str.count('}')
    open_brackets = json_str.count('[') - json_str.count(']')
    
    json_str += ']' * max(0, open_brackets)
    json_str += '}' * max(0, open_braces)
    return json_str

def safe_load_jsonl(filepath: Path) -> List[Dict[str, Any]]:
    records = []
    if not filepath.exists():
        print(f"❌ [Error] 파일을 찾을 수 없습니다: {filepath}")
        return records

    fixed_count = 0
    with open(filepath, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f, start=1):
            line_str = line.strip()
            if not line_str:
                continue
            try:
                records.append(json.loads(line_str))
            except json.JSONDecodeError:
                fixed_line = fix_truncated_json(line_str)
                try:
                    records.append(json.loads(fixed_line))
                    fixed_count += 1
                except Exception as e:
                    print(f"❌ [Line {idx}] 복구 실패: {e}")
                    
    if fixed_count > 0:
        print(f"🛠️ 잘린 JSON 구문 {fixed_count}건 복구 완료 ({filepath.name})")
    return records

testset_all = safe_load_jsonl(EVAL_DATASET_PATH)
chunks_all = safe_load_jsonl(CRAWLED_DOCS_PATH)

print(f"\n📊 로드된 평가 질문 (testset_all): {len(testset_all)}건")
print(f"📚 로드된 청크 데이터 (chunks_all): {len(chunks_all)}건")


📊 로드된 평가 질문 (testset_all): 580건
📚 로드된 청크 데이터 (chunks_all): 494건


In [5]:
import torch
import gc
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer

class VectorSearchEngine:
    def __init__(self, model_name: str, device: str = None, max_seq_length: int = 1024):
        self.model_name = model_name
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"🚀 [{model_name}] 모델 로딩 중... (Device: {self.device})")
        
        # 이전 메모리 찌꺼기 완벽 제거
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # bfloat16 정밀도 설정
        model_kwargs = {}
        if torch.cuda.is_available():
            model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

        self.model = SentenceTransformer(
            model_name, 
            device=self.device, 
            trust_remote_code=True,
            model_kwargs=model_kwargs
        )
        
        # 🔑 [핵심 해결책] 8B 모델의 메모리 폭발을 막기 위해 최대 토큰 길이를 제한합니다.
        self.model.max_seq_length = max_seq_length
        print(f"📏 최대 시퀀스 길이 제한: {self.model.max_seq_length} tokens")

        self.chunk_ids = []
        self.embeddings = None

    def build_index(self, chunks: List[Dict[str, Any]], batch_size: int = 4):
        texts = []
        self.chunk_ids = []
        
        for idx, item in enumerate(chunks):
            c_id = item.get('chunk_id') or item.get('doc_id') or item.get('id') or f"doc_{idx}"
            c_text = item.get('content') or item.get('text') or item.get('page_content') or ""
            
            self.chunk_ids.append(str(c_id))
            texts.append(c_text)

        print(f"⚡ [{self.model_name}] {len(texts)}개 청크 임베딩 계산 중... (Batch Size: {batch_size})")
        
        # 🔑 추론 모드로 실행 및 배치 사이즈 축소(batch_size=4)
        with torch.no_grad():
            self.embeddings = self.model.encode(
                texts, 
                batch_size=batch_size, 
                show_progress_bar=True, 
                convert_to_tensor=True, 
                normalize_embeddings=True
            )

    def search(self, query: str, top_k: int = 10) -> List[str]:
        with torch.no_grad():
            query_vec = self.model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
            scores = torch.matmul(self.embeddings, query_vec)
            top_indices = torch.topk(scores, k=min(top_k, len(self.chunk_ids))).indices.tolist()
        return [self.chunk_ids[i] for i in top_indices]

    def clear_memory(self):
        """다음 8B 모델 로드를 위한 VRAM 완전 초기화"""
        del self.model
        if self.embeddings is not None:
            del self.embeddings
        self.embeddings = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("✅ OOM 방지 최적화 VectorSearchEngine 준비 완료")

✅ OOM 방지 최적화 VectorSearchEngine 준비 완료


In [6]:
from tqdm import tqdm

class RAGEvaluator:
    @staticmethod
    def evaluate(search_engine: VectorSearchEngine, test_dataset: List[Dict[str, Any]], k_list=[1, 3, 5]) -> Dict[str, float]:
        metrics = {f'Recall@{k}': 0.0 for k in k_list}
        metrics.update({f'HitRate@{k}': 0.0 for k in k_list})
        metrics['MRR'] = 0.0
        
        valid_q_count = 0

        for q in tqdm(test_dataset, desc=f"Evaluating {search_engine.model_name.split('/')[-1]}"):
            query_text = q.get('question') or q.get('query') or ""
            expected = q.get('expected_sources', [])
            if isinstance(expected, str):
                expected = [expected]
            expected_set = set(expected)

            if not query_text or not expected_set:
                continue

            valid_q_count += 1
            retrieved_ids = search_engine.search(query_text, top_k=max(k_list))

            # MRR 계산
            for rank, r_id in enumerate(retrieved_ids, start=1):
                if any(r_id == exp or r_id.startswith(str(exp)) for exp in expected_set):
                    metrics['MRR'] += 1.0 / rank
                    break

            # Recall@K 및 HitRate@K
            for k in k_list:
                top_k_retrieved = retrieved_ids[:k]
                hits = sum(1 for r_id in top_k_retrieved if any(r_id == exp or r_id.startswith(str(exp)) for exp in expected_set))
                
                if hits > 0:
                    metrics[f'HitRate@{k}'] += 1.0
                metrics[f'Recall@{k}'] += min(1.0, hits / len(expected_set))

        if valid_q_count > 0:
            for key in metrics:
                metrics[key] = round(metrics[key] / valid_q_count, 4)

        return metrics

print("✅ RAGEvaluator 준비 완료")

✅ RAGEvaluator 준비 완료


In [7]:
import pandas as pd

# 지정해주신 3개 모델 (Hugging Face 리포지토리명)
CANDIDATE_MODELS = [
    "Qwen/Qwen3-Embedding-8B",
    "nvidia/Nemotron-3-Embed-8B-BF16",
    "dragonkue/BGE-m3-ko"
]

experiment_results = {}

if testset_all and chunks_all:
    for model_name in CANDIDATE_MODELS:
        print("\n" + "="*60)
        print(f"🧪 모델 실험 시작: {model_name}")
        print("="*60)
        
        try:
            # max_seq_length를 1024로 제한하여 A100 VRAM 안전지대 확보
            engine = VectorSearchEngine(model_name, max_seq_length=1024)
            engine.build_index(chunks_all, batch_size=4)
            
            results = RAGEvaluator.evaluate(engine, testset_all, k_list=[1, 3, 5])
            experiment_results[model_name] = results
            
            # 모델 종료 시 즉시 VRAM 비우기
            engine.clear_memory()
            print(f"🧹 [{model_name}] VRAM 비우기 완료")
            
        except Exception as e:
            print(f"❌ [{model_name}] 실행 중 오류 발생: {e}")

    # 최종 결과 표 작성
    if experiment_results:
        df_results = pd.DataFrame(experiment_results).T
        df_results.index.name = "Embedding Model"
        
        formatted_df = df_results.copy()
        for col in formatted_df.columns:
            formatted_df[col] = formatted_df[col].apply(lambda x: f"{x * 100:.2f}%")

        print("\n" + "📊 "*5 + "지정 모델 3종 최종 성능 비교 표" + " 📊"*5)
        display(formatted_df)
else:
    print("❌ 데이터가 로드되지 않았습니다. Cell 1/2를 확인해 주세요.")


🧪 모델 실험 시작: Qwen/Qwen3-Embedding-8B
🚀 [Qwen/Qwen3-Embedding-8B] 모델 로딩 중... (Device: cuda)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

📏 최대 시퀀스 길이 제한: 1024 tokens
⚡ [Qwen/Qwen3-Embedding-8B] 494개 청크 임베딩 계산 중... (Batch Size: 4)


Batches:   0%|          | 0/124 [00:00<?, ?it/s]

...

📏 최대 시퀀스 길이 제한: 1024 tokens
⚡ [dragonkue/BGE-m3-ko] 494개 청크 임베딩 계산 중... (Batch Size: 4)


Batches:   0%|          | 0/124 [00:00<?, ?it/s]

Evaluating BGE-m3-ko: 100%|██████████| 580/580 [00:11<00:00, 49.08it/s]


🧹 [dragonkue/BGE-m3-ko] VRAM 비우기 완료

📊 📊 📊 📊 📊 지정 모델 3종 최종 성능 비교 표 📊 📊 📊 📊 📊


,Recall@1,Recall@3,Recall@5,HitRate@1,HitRate@3,HitRate@5,MRR
Embedding Model,,,,,,,
Qwen/Qwen3-Embedding-8B,64.99%,81.69%,88.15%,64.99%,81.69%,88.15%,73.92%
nvidia/Nemotron-3-Embed-8B-BF16,64.09%,83.66%,88.33%,64.09%,83.66%,88.33%,73.99%
dragonkue/BGE-m3-ko,65.35%,84.74%,90.13%,65.35%,84.74%,90.13%,75.17%
